# 6강 실습: 수치 연산과 배열

`data/apicius/index.tsv`와 `word-freq.txt`로 5강의 값을 numpy로 다시 구한다.

- 이 노트북은 저장소 **맨 위**(`/workspaces/LDS2026`)에 두어야 `data/apicius/…` 경로가 맞는다.
- 쓰는 파일은 둘이다. `index.tsv`는 4강의 `awk/index.awk`로, `word-freq.txt`는 3강에서 만들었다.
- 처음 한 번은 터미널에서 `pip install numpy`를 실행한다.
- 커밋하기 전에는 [Restart] 후 [Run All]로 위에서 아래까지 한 번에 돌려 본다.

## 0. 준비

노트북이 **어느 폴더에서** 실행되는지 확인한다. `os.getcwd()`는 현재 작업 폴더(current working directory)를 돌려준다.

- 셀의 **마지막 식**은 `print` 없이도 결과가 저절로 표시된다. 여러 값을 함께 보려면 `print`를 쓴다.
- 결과가 `'/workspaces/LDS2026'`이 아니면 노트북의 위치가 잘못된 것이다. 아래 셀의 `data/apicius/…` 경로는 모두 이 폴더를 기준으로 한다.

In [1]:
import os
os.getcwd()

'/workspaces/LDS2026'

## 1. 순수 Python의 한계

numpy 없이 Python만으로 `words` 열(셋째 열)의 평균을 구한다.

- `with open(...) as f`: 파일을 열고, 블록이 끝나면 저절로 닫는다.
- `next(f)`: 첫 줄(머리글)을 읽어 버린다. AWK의 `NR > 1`에 해당한다.
- `line.split('\t')[2]`: 탭으로 나눈 조각 중 셋째다(0부터 센다). 글자이므로 `int(...)`로 정수로 바꾼다.

답은 5강과 같지만 다섯 줄이 필요했고, 표준편차까지 구하려면 코드가 더 길어진다.

In [2]:
nums = []
with open('data/apicius/index.tsv') as f:
    next(f)                       # 머리글 행 건너뛰기 (AWK의 NR > 1)
    for line in f:
        nums.append(int(line.split('\t')[2]))

print(len(nums), sum(nums) / len(nums))

499 45.124248496993985


리스트끼리 `+`를 하면 원소끼리 더해질까?

리스트의 `+`는 **이어 붙이기**다. 같은 까닭으로 `[1, 2, 3] * 2`는 두 배가 아니라 두 번 반복한 `[1, 2, 3, 1, 2, 3]`이 되고, `[1, 2, 3] + 1`이나 `[1, 2, 3] * 1.5`는 `TypeError`가 난다. 리스트의 `+`와 `*`는 수의 연산이 아니라 줄 세우기다.

In [3]:
[1, 2, 3] + [10, 20, 30]

[1, 2, 3, 10, 20, 30]

같은 계산을 numpy **배열**로 한다.

- `import numpy as np`: numpy를 `np`라는 짧은 이름으로 불러온다. 전 세계가 지키는 관례다.
- `np.array([...])`: 리스트를 배열로 바꾼다.
- 배열끼리의 `+`는 **같은 자리의 원소끼리** 더한다. 배열 전체에 한 번에 적용되는 이런 원소별 연산을 **벡터화 연산**이라 한다.

In [4]:
import numpy as np

a = np.array([1, 2, 3])
b = np.array([10, 20, 30])
a + b

array([11, 22, 33])

배열에 수를 곱하면 원소마다 곱해진다. 리스트의 `* 2`는 두 번 반복이었지만, 배열의 `* 2`는 **두 배**다.

In [5]:
a * 2

array([2, 4, 6])

리스트와 배열의 속도를 잰다. 100만 개의 수를 각각 두 배로 만든다.

- `1_000_000`: 밑줄은 읽기 쉽게 끊어 쓴 것일 뿐, `1000000`과 같다.
- `np.arange(n)`: 0부터 n−1까지의 정수 배열. Python의 `range`와 같은 규칙이다.
- `[x * 2 for x in nums]`: 리스트 컴프리헨션. 원소를 하나씩 꺼내 두 배로 만든 새 리스트다.
- `%timeit`: Jupyter가 주는 측정 도구. 여러 번 돌려 평균 시간과 표준편차를 알려 준다. Python 문법이 아니라서 `.py` 파일에서는 쓸 수 없다.

`ms`는 1000분의 1초, `μs`는 100만분의 1초다. 배열 쪽이 수십 배 빠르다. 값은 기계마다 다르니 여러분의 결과가 이 노트북과 달라도 정상이다.

In [6]:
nums = list(range(1_000_000))
arr = np.arange(1_000_000)

%timeit [x * 2 for x in nums]
%timeit arr * 2

17 ms ± 157 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


490 μs ± 6.48 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


## 2. 배열 만들기

배열이 스스로 알고 있는 네 가지 속성이다.

- `shape`: 모양. 각 축의 길이를 담은 튜플이다. `(5,)`는 길이 5인 1차원이고, 쉼표는 원소가 하나인 튜플의 표기다.
- `dtype`: 원소의 자료형. `int64`는 8바이트 정수다.
- `size`: 원소의 총 개수.
- `ndim`: 축의 개수(차원).

In [7]:
a = np.array([144, 82, 73, 82, 23])
print(a.shape, a.dtype, a.size, a.ndim)

(5,) int64 5 1


한 배열에는 **한 가지 자료형**만 담긴다. 섞어 넣으면 더 넓은 자료형으로 끌려간다.

- 정수만 넣으면 `int64` 배열이 된다.
- 정수와 실수를 섞으면 전부 실수(`float64`)가 된다. `1`이 `1.`이 된다.
- 문자열이 하나라도 있으면 전부 문자열이 된다. `dtype='<U21'`은 최대 21글자짜리 유니코드 문자열이라는 뜻이다.

수 `1`이 글자 `'1'`로 바뀌면 더 이상 더할 수 없다. `dtype`은 항상 확인한다.
아래 세 줄은 **한 줄씩 따로** 실행한다. 세 줄을 한 셀에 넣으면 마지막 결과만 보인다. 셀은 마지막 식만 저절로 표시하기 때문이다.

In [8]:
np.array([1, 2, 3])

array([1, 2, 3])

In [9]:
np.array([1, 2.5])

array([1. , 2.5])

In [10]:
np.array([1, 'PEPPER', 3])

array(['1', 'PEPPER', '3'], dtype='<U21')

배열을 처음부터 만들어 내는 함수들이다.

- `np.arange(5)`, `np.arange(0, 10, 2)`: `range`와 같은 규칙이다. 끝값은 들어가지 않는다.
- `np.zeros(3)`, `np.ones(3)`: 0 또는 1로 채운 배열. `0.`처럼 보이는 것은 `float64`이기 때문이다.
- `np.linspace(시작, 끝, 개수)`: **끝값을 포함해** 같은 간격으로 나눈다.

여기서도 다섯 줄을 **한 줄씩 따로** 실행한다.

In [11]:
np.arange(5)

array([0, 1, 2, 3, 4])

In [12]:
np.arange(0, 10, 2)

array([0, 2, 4, 6, 8])

In [13]:
np.zeros(3)

array([0., 0., 0.])

In [14]:
np.ones(3)

array([1., 1., 1.])

In [15]:
np.linspace(0, 1, 5)

array([0.  , 0.25, 0.5 , 0.75, 1.  ])

## 3. 파일에서 배열로

`index.tsv`의 `words` 열을 배열로 읽는다.

- `delimiter='\t'`: 필드를 나누는 글자. AWK의 `FS`에 해당한다.
- `skiprows=1`: 앞의 한 행(머리글)을 건너뛴다. AWK의 `NR > 1`에 해당한다.
- `usecols=2`: 셋째 열만 읽는다. **0부터 세므로** 2가 셋째다.

`np.loadtxt`는 값을 모두 `float64`로 읽는다. 그래서 `144`가 `144.`로 보인다.
`usecols`를 빼면 `title` 열의 글자를 수로 바꾸지 못해 `ValueError`가 난다. 배열은 한 자료형만 담기 때문이다. AWK는 한 표에 수와 글자가 섞여 있어도 아무렇지 않았다.

In [16]:
words = np.loadtxt('data/apicius/index.tsv', delimiter='\t',
                   skiprows=1, usecols=2)
print(words.shape, words.dtype)
print(words[:5])

(499,) float64
[144.  82.  73.  82.  23.]


`number` 열과 `title` 열은 따로 읽는다. 글자 열은 `dtype=str`을 주어야 한다.

한 표를 배열 **세 개**(`numbers`, `titles`, `words`)로 쪼개 들고 있게 되었다. 셋의 순서가 서로 맞는다는 것은 우리가 기억할 뿐, 배열은 모른다. 이 불편을 푸는 것이 7강의 데이터프레임이다.

In [17]:
numbers = np.loadtxt('data/apicius/index.tsv', delimiter='\t',
                     skiprows=1, usecols=0)
titles = np.loadtxt('data/apicius/index.tsv', delimiter='\t',
                    skiprows=1, usecols=1, dtype=str)
print(titles[:3])

['FINE SPICED WINE' 'HONEY REFRESHER FOR TRAVELERS' 'ROMAN VERMOUTH']


## 4. 인덱싱과 슬라이싱

대괄호 안에 **자리 번호**(색인)를 넣어 원소 하나를 꺼낸다.

- 번호는 **0부터** 센다. `words[0]`이 1번 레시피, `words[219]`가 220번 레시피다. 레시피 번호와 색인이 하나씩 어긋난다.
- 음수는 뒤에서부터 센다. `words[-1]`은 마지막 레시피다.
- `words[219]`가 0인 것이 5강에서 본 그 220번 레시피다. 제목과 본문 사이에 빈 줄이 없어서 본문이 비어 버렸다.

마지막 줄 `words[0]`은 `print` 없이 셀의 마지막 식으로 두었다. numpy 2부터 스칼라를 이렇게 표시하면 `np.float64(144.0)`처럼 나온다. 첫 줄처럼 `print`로 감싸면 값만 보인다. 한 셀 안에서 두 표시를 견주어 볼 수 있다. 그래서 이 노트북은 대부분 `print`를 쓴다.

In [18]:
print(words[0])
print(words[-1])
print(words[219])
words[0]

144.0
25.0
0.0


np.float64(144.0)

`[시작:끝:간격]`으로 여러 개를 한꺼번에 꺼낸다. 문자열·리스트와 같은 문법이고, 끝값은 들어가지 않는다.

- `words[:5]`: 처음 다섯 개
- `words[-3:]`: 마지막 세 개
- `words[::100]`: 100개마다 하나씩 — 색인 0, 100, 200, 300, 400
- `words[[0, 218, 219]]`: 색인을 **목록으로** 주면 원하는 자리만 고른다

네 줄을 **한 줄씩 따로** 실행한다.

In [19]:
words[:5]

array([144.,  82.,  73.,  82.,  23.])

In [20]:
words[-3:]

array([47., 29., 25.])

In [21]:
words[::100]

array([144.,  17.,  82.,  69.,   8.])

In [22]:
words[[0, 218, 219]]

array([144.,  11.,   0.])

슬라이스는 복사본이 아니다.

`b = a[:3]`은 `a`의 앞 세 칸을 **들여다보는 창**(뷰, view)이다. 그래서 `b[0]`을 바꾸면 `a[0]`도 99로 바뀐다. 원본을 지키려면 `b = a[:3].copy()`처럼 분명히 복사한다. 리스트의 슬라이스는 복사본이므로, 리스트와 달라지는 몇 안 되는 규칙이다.

In [23]:
a = np.arange(10)
b = a[:3]          # 잘라낸 것처럼 보이지만
b[0] = 99          # 원본이 함께 바뀐다
print(a)

[99  1  2  3  4  5  6  7  8  9]


## 5. 벡터화 연산

스칼라 하나와 배열을 계산하면 그 스칼라가 배열 전체에 퍼진다. 이것을 **브로드캐스팅**이라 한다. `words / 100`은 499개 전부를 100으로 나눈다.

여기서는 각 레시피가 책 전체(`words.sum()`, 22,517단어)의 몇 %인지 구한다(**정규화**). 길이가 다른 자료를 견줄 때 쓴다.
`np.round(값, 2)`는 소수점 아래 둘째 자리까지 반올림한다.

In [24]:
print(np.round(words[:5] / words.sum() * 100, 2))

[0.64 0.36 0.32 0.36 0.1 ]


평균에서 표준편차 몇 개만큼 떨어져 있는지를 구한다(**표준화**, $z$-점수).

$$z_i = \frac{x_i - \bar{x}}{s}$$

`words - words.mean()`은 499개 각각에서 평균을 빼고, 그 결과를 표준편차로 나눈다. 1번 레시피의 2.84는 평균보다 표준편차 2.84개만큼 길다는 뜻이다. 단위(단어)가 사라지므로 다른 변수와도 견줄 수 있다.

In [25]:
z = (words - words.mean()) / words.std(ddof=1)
print(np.round(z[:5], 2))

[ 2.84  1.06  0.8   1.06 -0.64]


## 6. 집계

5강에서 AWK로 구한 기술통계량을 numpy로 다시 구한다.

- `size`, `sum()`: 개수와 합계
- `mean()`, `np.median()`, `std(ddof=1)`: 평균, 중앙값, 표준편차
- `min()`, `max()`: 최솟값과 최댓값

5강의 값은 `499 22517 45.1242`, 중앙값 33, 표준편차 34.77, 최소·최대 0과 273이었다. 도구가 바뀌어도 답은 바뀌지 않는다.
대부분의 집계는 메서드(`words.mean()`)와 함수(`np.mean(words)`)가 모두 있지만, 중앙값과 분위수는 `np.median`, `np.percentile` 함수만 있다.

In [26]:
print(words.size, words.sum())
print(words.mean(), np.median(words), words.std(ddof=1))
print(words.min(), words.max())

499 22517.0
45.124248496993985 33.0 34.77403524607915
0.0 273.0


`ddof`는 표준편차를 구할 때 $n$에서 **빼는 수**다(*delta degrees of freedom*).

- `std(ddof=1)`: $n-1$로 나눈다. 5강에서 쓴 식이다.
- `std()`: 기본값 `ddof=0`, 곧 $n$으로 나눈다.

numpy의 기본값은 $n$이므로 5강과 같은 값을 원하면 `ddof=1`을 **매번** 적는다. 언어학 논문의 관행은 $n-1$이라 이 수업에서는 `ddof=1`을 기본으로 한다.

In [27]:
print(words.std(ddof=1))
print(words.std())

34.77403524607915
34.7391740490991


`np.percentile`에 목록을 주면 여러 분위수를 한 번에 돌려준다. 0·25·50·75·100%는 최솟값·Q1·중앙값·Q3·최댓값, 곧 **다섯 수치 요약**이다.

5강에서 `sort`한 뒤 네 자리를 손으로 집어내던 파이프라인이 한 줄이 되었다. 값이 딱 떨어지지 않으면 이웃한 두 값 사이를 비례로 계산하므로, 자료에 따라 5강의 방법과 답이 다를 수 있다. 이 자료에서는 같다.

In [28]:
print(np.percentile(words, [0, 25, 50, 75, 100]))

[  0.  20.  33.  61. 273.]


## 7. 불리언 마스크

배열을 비교하면 결과도 배열이다. 원본과 **길이가 같은** `True`/`False` 배열을 **불리언 마스크**라 한다.

`words[:6] > words.mean()`은 여섯 레시피 각각이 평균(45.12)보다 긴지를 한꺼번에 따진다. 조건을 한 번 만들어 두면 세는 데도, 골라내는 데도 쓸 수 있다.

In [29]:
print(words[:6])
print(words[:6] > words.mean())

[144.  82.  73.  82.  23.  53.]
[ True  True  True  True False  True]


마스크로 **세기**. `True`는 1, `False`는 0으로 계산된다.

- `mask.sum()`: `True`의 개수, 곧 평균보다 짧은 레시피의 수
- `mask.mean()`: `True`의 비율

`f'{mask.mean():.1%}'`는 소수를 백분율로 적는 f-문자열 서식이다. `.1%`는 100을 곱해 소수점 아래 한 자리로 적고 `%`를 붙인다. AWK의 `printf`에 해당한다. 5강의 "레시피의 62.7%가 평균보다 짧다"가 한 줄로 다시 나온다.

In [30]:
mask = words < words.mean()
print(mask.sum(), f'{mask.mean():.1%}')

313 62.7%


마스크로 **골라내기**. 대괄호 안에 마스크를 넣으면 `True`인 자리만 남는다.

- `words[words > 200]`: 200단어가 넘는 레시피의 길이
- `(words >= 20) & (words <= 61)`: 두 조건을 **원소별로** 잇는다. Q1과 Q3 사이, 곧 가운데 절반에 드는 레시피를 센다.

배열에는 `and`, `or`, `not` 대신 `&`(그리고), `|`(또는), `~`(아니다)를 쓴다. 이 기호들은 비교 연산자보다 먼저 계산되므로 **각 조건을 괄호로 감싼다**. `and`를 쓰면 `ValueError: The truth value of an array ...`가 난다.

In [31]:
print(words[words > 200])
print(((words >= 20) & (words <= 61)).sum())   # 가운데 절반

[206. 273.]
258


`words`로 만든 마스크를 **다른 배열**(`numbers`, `titles`)에 그대로 쓴다. 세 배열의 길이와 순서가 같기 때문에 가능하다.

- `words == 0`: 0단어인 레시피 → 220번
- `words.argmax()`: 최댓값의 **자리**(색인)를 준다. `max()`는 값을 준다. → 가장 긴 186번 레시피

5강의 "최솟값과 최댓값은 눈으로 확인한다"가 한 줄이 되었다.

In [32]:
print(numbers[words == 0], titles[words == 0])
print(numbers[words.argmax()], titles[words.argmax()])

[220.] ['FOR ROASTS: PEPPER, LOVAGE, CORIANDER, CARRAWAY,']
186.0 PEAS [supreme style]


## 8. 기술통계량을 직접 만들기

5강의 표준편차 식을 그대로 옮겨, `std(ddof=1)`과 같은 값이 나오는지 확인한다.

$$s = \sqrt{\frac{\sum (x_i-\bar{x})^2}{n-1}}$$

- `d = words - words.mean()`: 편차 $x_i-\bar{x}$를 499개 한꺼번에 구한다.
- `(d ** 2).sum()`: 편차 제곱의 합. `**`는 거듭제곱이다.
- `np.sqrt(...)`: 제곱근

수식과 코드가 한 줄씩 대응한다. 5강의 `awk/stats.awk`와 달리 `for`문도, 값을 담아 둘 배열도 필요 없다.

In [33]:
d = words - words.mean()          # 편차
ss = (d ** 2).sum()               # 편차 제곱의 합
print(np.sqrt(ss / (words.size - 1)))
print(words.std(ddof=1))          # 맞는지 확인

34.77403524607915
34.77403524607915


편차의 합은 이론상 0이어야 한다. 그런데 결과는 `2.2737367544323206e-12`다. `e-12`는 $\times 10^{-12}$이니 0.0000000000022737이다.

컴퓨터는 실수를 정해진 자릿수로만 기억하므로 499번 더하는 동안 아주 작은 오차가 쌓였다. 실수끼리는 `==` 대신 `np.isclose(d.sum(), 0)`으로 묻는다. 앞에서 쓴 `words == 0`이 안전했던 것은 단어를 **센** 정수이기 때문이다.

In [34]:
print(d.sum())

2.2737367544323206e-12


## 9. 길이와 빈도

3강에서 만든 빈도 목록 `word-freq.txt`를 읽는다. 한 줄에 빈도와 단어가 공백으로 나뉘어 있다.

- `delimiter`를 적지 않으면 **공백**으로 나눈다. `sort | uniq -c`가 만든 파일이라 줄 앞에 공백이 붙어 있는데, 알아서 처리된다.
- 빈도(`usecols=0`)는 수로, 단어(`usecols=1`)는 `dtype=str`로 따로 읽는다.

2,434개 단어가 빈도순으로 들어 있다. 1위는 1,265번 나온 `AND`다.

In [35]:
freq = np.loadtxt('data/apicius/word-freq.txt', usecols=0)
vocab = np.loadtxt('data/apicius/word-freq.txt', usecols=1, dtype=str)
print(freq.shape, freq[:5])
print(vocab[:5])

(2434,) [1265. 1009.  726.  467.  400.]
['AND' 'THE' 'WITH' 'PEPPER' 'BROTH']


단어마다 글자 수를 센다.

`[len(w) for w in vocab]`는 리스트 컴프리헨션이다. 단어를 하나씩 꺼내 `len`으로 길이를 잰 새 리스트를 만들고, `np.array`로 배열로 바꾼다.
numpy의 벡터화는 **수치** 연산을 위한 것이라 문자열에는 `for`가 다시 필요하다. 여기가 numpy의 경계다. 7강의 pandas에서는 이것이 `.str.len()` 한 번으로 된다.

In [36]:
length = np.array([len(w) for w in vocab])
print(length[:5])

[3 3 4 6 5]


단어의 길이와 빈도의 상관계수를 구한다.

- `np.corrcoef(x, y)`는 값 하나가 아니라 2×2 **상관행렬**을 돌려준다. 대각선은 자기 자신과의 상관이라 언제나 1이고, 원하는 값은 `[0, 1]` 자리다.
- `np.log10(freq)`: 2,434개 빈도 전부에 상용로그를 한 번에 취한다.

5강의 `awk/corr.awk`, `awk/corr-log.awk`와 같은 값(−0.1383, −0.2949)이 나온다. 13줄짜리 스크립트가 한 줄이 되었다.

In [37]:
print(np.corrcoef(length, freq)[0, 1])
print(np.corrcoef(length, np.log10(freq))[0, 1])

-0.13834537169094618
-0.2949203774523119


`freq`로 만든 마스크를 `length`에 씌워 두 집단의 평균 길이를 견준다.

- `freq >= 100`: 100번 이상 나온 단어
- `freq == 1`: 딱 한 번 나온 단어
- `:.2f`: 소수점 아래 두 자리로 적는 서식

자주 쓰는 말은 4.72글자, 한 번만 쓰인 말은 6.88글자다. 자주 쓰는 말일수록 짧다는 **Zipf의 약어 법칙**이 $r=-0.14$라는 밋밋한 수보다 잘 보인다. 5강의 "나누어 보라"가 이것이다.

In [38]:
print(f'{length[freq >= 100].mean():.2f}')   # 100번 이상 나온 단어
print(f'{length[freq == 1].mean():.2f}')     # 딱 한 번 나온 단어

4.72
6.88


딱 한 번 나온 단어(**단발어**, *hapax legomenon*)의 수와 비율이다.

2,434개 중 1,118개(45.9%)가 한 번만 나온다. 코퍼스를 아무리 키워도 이 비율은 좀처럼 줄지 않는다. "이 코퍼스에 없다"가 "그런 말이 없다"는 뜻이 아닌 이유이고, 8강에서 어휘의 생산성을 재는 재료가 된다.

In [39]:
print((freq == 1).sum(), f'{(freq == 1).mean():.1%}')

1118 45.9%


## 10. 2차원 배열

`usecols`에 열을 둘 주면 **2차원 배열**이 된다.

- `shape`가 `(499, 2)`, 곧 499행 2열이다. 앞이 행, 뒤가 열이다.
- 대괄호가 두 겹인 것이 2차원이라는 표시다.

`number`와 `words`는 둘 다 수라서 한 배열에 담을 수 있다. `title`은 넣을 수 없다.

In [40]:
table = np.loadtxt('data/apicius/index.tsv', delimiter='\t',
                   skiprows=1, usecols=(0, 2))
print(table.shape, table.ndim)
print(table[:3])

(499, 2) 2
[[  1. 144.]
 [  2.  82.]
 [  3.  73.]]


2차원 배열에서 값을 꺼내고 접기.

- `table[0, 1]`: 0행 1열의 값 하나. 쉼표로 두 축을 함께 쓴다.
- `table[:, 1]`: 모든 행(`:`)의 1열. 앞의 `words`와 같은 배열이다.
- `table.mean(axis=0)`: 행 방향으로 접는다. 열마다 평균이 하나씩 남는다.

번호 열의 평균 250은 아무 뜻이 없다. 5강의 척도 이야기 그대로, 계산이 된다고 뜻이 있는 것은 아니다. 열마다 자료형이 다른 표를 표인 채로 다루는 도구가 7강의 pandas다.

In [41]:
print(table[0, 1])
print(table[:, 1][:3])
print(table.mean(axis=0))

144.0
[144.  82.  73.]
[250.         45.1242485]
